# SQL filtering and grouping

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-01-filtering-and-grouping.ipynb)

Filtering decides *which rows* a query looks at. Grouping decides *what one result row means*. Most wrong
SQL answers come from mixing those two up, so this notebook covers both with a small, checkable dataset.

Start with the [index](../03_sql_interview_questions.ipynb) if you have not, and continue with
[WHERE vs HAVING](sql-02-where-vs-having.ipynb) after this one.

## Setup

The first code cells install PySpark if it is missing, detect Java, and start a local Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4", "setuptools": "setuptools>=70"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
# Detect PySpark and Java. Spark needs a Java runtime that can actually start a JVM.
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=60)
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
if not SPARK_AVAILABLE:
    print("SKIPPED: Spark is not available, so every example prints SKIPPED instead of running.")

In [ ]:
import os
import sys
import time
from datetime import date, datetime
from decimal import Decimal

# Spark timestamps and Python datetimes must agree on the clock, so pin the process to UTC.
os.environ["TZ"] = "UTC"
if hasattr(time, "tzset"):
    time.tzset()

if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("sql-learning-path")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session ready:", spark.version)
else:
    spark = None


def create_view(name, ddl, rows):
    """Register rows as a temporary view. Skipped when Spark is unavailable."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED view {name}: needs a Spark session.")
        return
    spark.createDataFrame(rows, ddl).createOrReplaceTempView(name)
    print(f"view {name}: {spark.table(name).count()} rows")


def _plain(value):
    """Spark returns DECIMAL for literal arithmetic; compare those as floats."""
    return float(value) if isinstance(value, Decimal) else value


def check(query, expected, label):
    """Show a query's result, then assert it equals the expected rows. Skipped without Spark."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED {label}: needs a Spark session.")
        return
    frame = spark.sql(query)
    frame.show(truncate=False)
    rows = [tuple(_plain(value) for value in row) for row in frame.collect()]
    assert rows == expected, f"{label}: got {rows}, expected {expected}"
    print(f"PASS {label}")

## The example data

Two tables drive the notebook: `customers` (one row per customer) and `orders` (one row per order).
Each is registered as a Spark temporary view, so the SQL reads like any warehouse table without any files.

In [ ]:
create_view(
    "customers",
    "customer_id INT, name STRING, region STRING, signup_date DATE",
    [
        (1, "Acme Corp", "North", date(2022, 1, 5)),
        (2, "Beta LLC", "North", date(2022, 2, 10)),
        (3, "Cyan Inc", "South", date(2022, 3, 15)),
        (4, "Delta Co", "South", date(2022, 4, 20)),
        (5, "Echo Ltd", "West", date(2022, 5, 25)),
        (6, "Foxtrot GmbH", "West", date(2022, 6, 30)),
        (7, "Gamma SA", "East", date(2023, 1, 10)),
        (8, "Hotel BV", "East", date(2023, 2, 15)),
    ],
)

create_view(
    "orders",
    "order_id INT, customer_id INT, status STRING, amount INT, order_date DATE",
    [
        (1, 1, "completed", 100, date(2023, 1, 5)),
        (2, 1, "completed", 200, date(2023, 2, 10)),
        (3, 1, "completed", 150, date(2023, 3, 15)),
        (4, 2, "completed", 300, date(2023, 1, 20)),
        (5, 2, "cancelled", 250, date(2023, 2, 5)),
        (6, 2, "completed", 400, date(2023, 3, 20)),
        (7, 3, "completed", 120, date(2023, 1, 25)),
        (8, 3, "completed", 180, date(2023, 2, 28)),
        (9, 4, "pending", 220, date(2023, 2, 14)),
        (10, 4, "completed", 320, date(2023, 3, 30)),
        (11, 5, "completed", 500, date(2023, 1, 2)),
        (12, 5, "completed", 450, date(2023, 3, 5)),
        (13, 6, "completed", 275, date(2023, 2, 18)),
        (14, 6, "cancelled", 325, date(2023, 3, 25)),
    ],
)

## How a query is evaluated

A SQL statement is written in one order but evaluated in another:

1. `FROM` and `JOIN` build the input rows.
2. `WHERE` keeps only rows that satisfy a condition, *before* any grouping.
3. `GROUP BY` collapses the remaining rows into one row per group.
4. `HAVING` keeps only groups that satisfy a condition.
5. `SELECT` computes the output columns, and `ORDER BY` sorts them.

Two consequences follow. A column alias from `SELECT` does not exist yet in `WHERE`, so you repeat the
expression. And a plain `WHERE` cannot see an aggregate, because the aggregate has not been computed yet.

## Filtering rows with `WHERE`

A predicate is a condition that is `TRUE`, `FALSE` or `NULL` for each row. `WHERE` keeps the `TRUE` rows.
The common operators are comparisons, `IN` for a list, `BETWEEN` for an inclusive range, and `LIKE` for
simple text patterns.

`BETWEEN` includes both ends. That is convenient for integers and dangerous for timestamps, where
`BETWEEN '2023-02-01' AND '2023-02-28'` silently misses everything after midnight on the 28th. Use a
half-open range (`>= start AND < end`) for timestamps; the time-series notebook shows why.

### Comparison and `BETWEEN`

Completed orders with an amount from 150 to 300, both ends included:

In [ ]:
check(
    """
    SELECT order_id, amount
    FROM orders
    WHERE status = 'completed' AND amount BETWEEN 150 AND 300
    ORDER BY order_id
    """,
    [(2, 200), (3, 150), (4, 300), (8, 180), (13, 275)],
    "completed orders between 150 and 300",
)

### `IN` for a list of values

Customers in two regions:

In [ ]:
check(
    """
    SELECT customer_id, name
    FROM customers
    WHERE region IN ('North', 'West')
    ORDER BY customer_id
    """,
    [(1, "Acme Corp"), (2, "Beta LLC"), (5, "Echo Ltd"), (6, "Foxtrot GmbH")],
    "customers in North or West",
)

## Grouping: one row per group

`GROUP BY` defines the **grain** of the result: one output row per distinct combination of the grouped
columns. Every other column in `SELECT` must either be grouped or wrapped in an aggregate such as `COUNT`,
`SUM`, `AVG`, `MIN` or `MAX`. Selecting `order_id` next to `GROUP BY status` is an error, because a status
has many orders and the engine cannot choose one.

Before you write the query, say the grain out loud: "one row per status" is a different result from
"one row per customer and status".

### Revenue by status

One row per status. `COUNT(*)` counts rows; `SUM` adds the amounts.

In [ ]:
check(
    """
    SELECT status, COUNT(*) AS orders, SUM(amount) AS gross_amount
    FROM orders
    GROUP BY status
    ORDER BY status
    """,
    [("cancelled", 2, 575), ("completed", 11, 2995), ("pending", 1, 220)],
    "orders and gross amount by status",
)

### Several aggregates per group

One query can compute as many aggregates as you need. `MIN` and `MAX` work on dates as well as numbers,
which makes them the usual tools for "first seen" and "largest".

In [ ]:
check(
    """
    SELECT customer_id,
           COUNT(*) AS orders,
           MIN(order_date) AS first_order,
           MAX(amount) AS largest_order
    FROM orders
    GROUP BY customer_id
    ORDER BY customer_id
    """,
    [
        (1, 3, date(2023, 1, 5), 200),
        (2, 3, date(2023, 1, 20), 400),
        (3, 2, date(2023, 1, 25), 180),
        (4, 2, date(2023, 2, 14), 320),
        (5, 2, date(2023, 1, 2), 500),
        (6, 2, date(2023, 2, 18), 325),
    ],
    "per-customer order summary",
)

## Check the grain

A grouped result is only trustworthy if the groups add back up to the source. Here the status counts must
sum to the 14 orders in the table. When this check fails after a change, a row was dropped or duplicated
upstream.

In [ ]:
check(
    """
    SELECT SUM(n) AS total_orders
    FROM (SELECT status, COUNT(*) AS n FROM orders GROUP BY status)
    """,
    [(14,)],
    "status groups add back to the table",
)

## Takeaways

- Filter with `WHERE` on row-level values; `HAVING` is for aggregates (next notebook).
- Decide the grain before you write `GROUP BY`.
- Use half-open ranges for timestamps, and check that grouped totals reconcile with the source.